In [ ]:
sampling_interval = '1h'

In [ ]:
data_path = "/home/lkapral/hb/data/"

In [ ]:
import numpy as np
import pandas as pd
import tqdm
import argparse
import os


tqdm.tqdm.pandas()

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_parquet(os.path.join(data_path, '!hb_chunks_15_rbc_combined.parquet'))

In [ ]:
df_raw = df.copy()

In [ ]:
df.iloc[5:6]

In [ ]:
ids = df['stay_id'].unique()

In [ ]:
mimic_vaso = pd.read_csv('/home/lkapral/hb/data/states_and_actions_filled.csv')

In [ ]:
mimic_vaso['icustayid'].nunique()

In [ ]:
mimic_vaso

In [ ]:
mimic_vaso.loc[mimic_vaso['Hb']>0,['icustayid', 'timestep', 'Hb']]

In [ ]:
mimic_vaso[['timestep','PEEP']].head(50)

In [ ]:
mimic= pd.read_csv('/home/lkapral/hb/data/patient_states.csv')

In [ ]:
mimic

In [ ]:
mimic[mimic.groupby('icustayid')['Hb'].transform('count') > 1]['icustayid'].nunique()

In [ ]:
mimic_vaso.drop(columns=['Hb'], inplace=True)

In [ ]:
# Merge the two datasets on icustayid and timestep
# Only bringing in input_step and output_step columns
mimic = mimic_vaso.merge(
    mimic[['icustayid', 'timestep','Hb']], 
    on=['icustayid', 'timestep'], 
    how='left'
)

In [ ]:
mimic

In [ ]:
mimic.loc[mimic['Hb']>0,:]

In [ ]:
df = df[df.groupby('stay_id')['avg_hgb'].transform('count') > 1]

In [ ]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. DETECT COLUMNS
# ---------------------------------------------------------

print("=== DF columns ===")
print(df.columns.tolist())

print("\n=== MIMIC columns ===")
print(mimic.columns.tolist())

# Find ID column in df (likely contains 'id', 'stay', 'encounter')
id_candidates_df = [c for c in df.columns if any(x in c.lower() for x in ['id', 'stay', 'encounter'])]
print(f"\nPossible ID columns in df: {id_candidates_df}")

# Find time column in df
time_candidates_df = [c for c in df.columns if any(x in c.lower() for x in ['time', 'step', 'hour', 'date'])]
print(f"Possible time columns in df: {time_candidates_df}")

In [ ]:
mimic['median_dose_vaso']

In [ ]:
mimic.loc[mimic['icustayid'].isin(ids),'output_step'].value_counts()

In [ ]:
mimic['Hb'].value_counts()

In [ ]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# FAST VECTORIZED FUSION
# ---------------------------------------------------------

print("=== FAST FUSION ===")

df_prep = df_raw.copy()
mimic_prep = mimic.copy()

# Create time bins
print("\n1. Creating time bins...")
df_prep['_bin'] = pd.to_datetime(df_prep['chartdate']).dt.floor('6h')
mimic_prep['_charttime'] = pd.to_datetime(mimic_prep['timestep'], unit='s')
mimic_prep['_bin'] = mimic_prep['_charttime'].dt.floor('6h')
mimic_prep = mimic_prep.rename(columns={'icustayid': 'stay_id'})

# ---------------------------------------------------------
# CREATE COMBINED VASO IN MIMIC (using formula)
# ---------------------------------------------------------

print("\n2. Creating combined vaso in mimic...")

# Vaso conversion factors (phenylephrine and epinephrine set to 0)
mimic_vaso_cols = {
    'norepinephrine': 1.0,
    'epinephrine': 0.0,        # Set to 0 - not used in training
    'vasopressin': 5.0,
    'phenylephrine': 0.0,      # Set to 0 - not used in training
    'dopamine': 0.0,
    'dobutamine': 0.01,
}

# Find matching columns in mimic (they might have different names)
mimic_vaso_mapping = {}
for drug in mimic_vaso_cols.keys():
    # Try to find the column
    matching_cols = [c for c in mimic_prep.columns if drug.lower() in c.lower()]
    if matching_cols:
        mimic_vaso_mapping[drug] = matching_cols[0]
        print(f"   Found {drug}: {matching_cols[0]}")

# Calculate combined vaso in mimic
mimic_prep['combined_vaso_mimic'] = 0.0
vaso_used_mimic = []

for drug, factor in mimic_vaso_cols.items():
    if factor == 0:
        continue  # Skip drugs with factor 0
    if drug in mimic_vaso_mapping:
        col = mimic_vaso_mapping[drug]
        mimic_prep['combined_vaso_mimic'] += mimic_prep[col].fillna(0) * factor
        vaso_used_mimic.append(col)

# Set to NaN if all vaso columns were NaN
if vaso_used_mimic:
    all_nan_mimic = pd.concat([mimic_prep[c].isna() for c in vaso_used_mimic], axis=1).all(axis=1)
    mimic_prep.loc[all_nan_mimic, 'combined_vaso_mimic'] = np.nan

print(f"   combined_vaso_mimic non-zero: {(mimic_prep['combined_vaso_mimic'] > 0).sum()}")

# ---------------------------------------------------------
# AGGREGATE MIMIC TO 6H BINS
# ---------------------------------------------------------

print("\n3. Aggregating mimic...")

agg_dict_mimic = {
    'Hb': 'mean',
    'Fibrinogen': 'mean',
    'Platelets_count': 'mean',
    'Arterial_lactate': 'max',
    'Arterial_BE': 'mean',
    'HR': 'mean',
    'SysBP': 'mean',
    'MeanBP': 'mean',
    'DiaBP': 'mean',
    'SpO2': 'mean',
    'RR': 'mean',
    'input_step': 'sum',
    'output_step': 'sum',
    'combined_vaso_mimic': 'mean',  # Use our calculated combined vaso
}

agg_dict_mimic = {k: v for k, v in agg_dict_mimic.items() if k in mimic_prep.columns}
mimic_agg = mimic_prep.groupby(['stay_id', '_bin']).agg(agg_dict_mimic).reset_index()

print(f"   mimic_agg: {len(mimic_agg)} unique bins")
print(f"   combined_vaso_mimic non-zero after agg: {(mimic_agg['combined_vaso_mimic'] > 0).sum()}")

# ---------------------------------------------------------
# CREATE COMBINED VASO IN DF (using same formula)
# ---------------------------------------------------------

print("\n4. Creating combined vaso in df...")

# Vaso conversion factors for df (phenylephrine and epinephrine set to 0)
df_vaso_cols = {
    'avg_norepinephrine': 1.0,
    'avg_epinephrine': 0.0,        # Set to 0 - not used in training
    'avg_vasopressin': 5.0,
    'avg_phenylephrine': 0.0,      # Set to 0 - not used in training
    'avg_dobutamine': 0.01,
}

df_prep['combined_vaso'] = 0.0
vaso_used_df = []

for col, factor in df_vaso_cols.items():
    if factor == 0:
        continue  # Skip drugs with factor 0
    if col in df_prep.columns:
        df_prep['combined_vaso'] += df_prep[col].fillna(0) * factor
        vaso_used_df.append(col)
        print(f"   Using {col} x {factor}")

# Set to NaN if all vaso columns were NaN
if vaso_used_df:
    all_nan_df = pd.concat([df_prep[c].isna() for c in vaso_used_df], axis=1).all(axis=1)
    df_prep.loc[all_nan_df, 'combined_vaso'] = np.nan

print(f"   combined_vaso non-zero: {(df_prep['combined_vaso'] > 0).sum()}")

# ---------------------------------------------------------
# AGGREGATE DF TO 6H BINS
# ---------------------------------------------------------

print("\n5. Aggregating df to 6h bins...")

agg_dict_df = {
    # IDs - take first
    'subject_id': 'first',
    'hadm_id': 'first',
    'gender': 'first',
    'round': 'first',
    'chartdate': 'first',
    'icu_intime': 'first',
    'icu_outtime': 'first',
    'window_end': 'first',
    'row_id': 'first',
    'index': 'first',
    
    # Labs - mean (except lactate=max, baseexcess=min)
    'avg_fibrinogen': 'mean',
    'avg_hgb': 'mean',
    'avg_platelet': 'mean',
    'max_lactate': 'max',
    'min_baseexcess': 'min',
    'min_hb_bga': 'min',
    
    # Vitals - mean
    'avg_sbp': 'mean',
    'avg_sbp_ni': 'mean',
    'avg_dbp': 'mean',
    'avg_dbp_ni': 'mean',
    'avg_mbp': 'mean',
    'avg_mbp_ni': 'mean',
    'avg_heart_rate': 'mean',
    'avg_spo2': 'mean',
    'avg_resp_rate': 'mean',
    
    # Vasopressors - mean (keep individual for reference, but use combined)
    'avg_dobutamine': 'mean',
    'avg_norepinephrine': 'mean',
    'avg_vasopressin': 'mean',
    'avg_epinephrine': 'mean',
    'avg_phenylephrine': 'mean',
    'combined_vaso': 'mean',
    
    # Fluids/outputs - sum
    'total_urineoutput': 'sum',
    'chest_pericardial_output': 'sum',
    'jp_penrose_output': 'sum',
    'general_drain_output': 'sum',
    'hemo_wound_output': 'sum',
    'crystalloid_input': 'sum',
    'colloid_input': 'sum',
    'blood_input': 'sum',
}

# Keep only columns that exist in df_prep
agg_dict_df = {k: v for k, v in agg_dict_df.items() if k in df_prep.columns}

# Aggregate df to 6h bins
df_6h = df_prep.groupby(['stay_id', '_bin']).agg(agg_dict_df).reset_index()

print(f"   df_6h: {len(df_6h)} unique bins (from {len(df_prep)} rows)")
print(f"   avg_hgb non-null: {df_6h['avg_hgb'].notna().sum()}")
print(f"   combined_vaso non-zero: {(df_6h['combined_vaso'] > 0).sum()}")

# ---------------------------------------------------------
# MERGE WITH MIMIC AND FILL MISSING VALUES
# ---------------------------------------------------------

print("\n6. Merging with mimic and filling missing values...")

# Column mapping (now using combined_vaso_mimic instead of median_dose_vaso)
column_pairs = [
    ('avg_hgb', 'Hb'),
    ('avg_fibrinogen', 'Fibrinogen'),
    ('avg_platelet', 'Platelets_count'),
    ('max_lactate', 'Arterial_lactate'),
    ('min_baseexcess', 'Arterial_BE'),
    ('avg_heart_rate', 'HR'),
    ('avg_sbp', 'SysBP'),
    ('avg_mbp', 'MeanBP'),
    ('avg_dbp', 'DiaBP'),
    ('avg_spo2', 'SpO2'),
    ('avg_resp_rate', 'RR'),
    ('crystalloid_input', 'input_step'),
    ('total_urineoutput', 'output_step'),
    ('combined_vaso', 'combined_vaso_mimic'),  # Use calculated combined vaso
]

allow_zero_negative = {'min_baseexcess', 'combined_vaso', 'crystalloid_input', 'total_urineoutput'}

# Merge df_6h with mimic_agg
df_merged = df_6h.merge(mimic_agg, on=['stay_id', '_bin'], how='left')

print(f"   Merged: {len(df_merged)} rows")
print(f"   Rows with mimic Hb: {df_merged['Hb'].notna().sum()}")
print(f"   Rows with mimic combined_vaso: {df_merged['combined_vaso_mimic'].notna().sum()}")

# Fill missing values
print("\n7. Filling missing values (df priority)...")

fill_stats = {}

for df_col, mimic_col in column_pairs:
    if df_col not in df_merged.columns or mimic_col not in df_merged.columns:
        print(f"   SKIP: {df_col} or {mimic_col} not found")
        continue
    
    before = df_merged[df_col].notna().sum()
    
    if df_col in allow_zero_negative:
        # Only check notna
        mask = df_merged[df_col].isna() & df_merged[mimic_col].notna()
    else:
        # Check notna AND > 0
        mask = (
            (df_merged[df_col].isna() | (df_merged[df_col] <= 0)) & 
            df_merged[mimic_col].notna() & 
            (df_merged[mimic_col] > 0)
        )
    
    df_merged.loc[mask, df_col] = df_merged.loc[mask, mimic_col]
    
    after = df_merged[df_col].notna().sum()
    filled = after - before
    fill_stats[df_col] = filled
    
    print(f"   {df_col}: {before} → {after} (+{filled})")

# ---------------------------------------------------------
# CLEANUP
# ---------------------------------------------------------

print("\n8. Cleanup...")

# Drop mimic columns
mimic_cols_to_drop = [col for col in agg_dict_mimic.keys() if col in df_merged.columns]
df_merged = df_merged.drop(columns=mimic_cols_to_drop, errors='ignore')

# Rename _bin to time_bin_6h
df_merged = df_merged.rename(columns={'_bin': 'time_bin_6h'})

print(f"   Final columns: {df_merged.columns.tolist()}")

# ---------------------------------------------------------
# SUMMARY
# ---------------------------------------------------------

print("\n" + "=" * 50)
print("SUMMARY")
print("=" * 50)

print(f"\nOriginal df_raw: {df_raw.shape}")
print(f"Final df (6h bins): {df_merged.shape}")

print("\nVaso formula used:")
print("  combined_vaso = norepinephrine * 1.0")
print("                + vasopressin * 5.0")
print("                + dobutamine * 0.01")
print("  (epinephrine and phenylephrine set to 0)")

print("\nValues filled from mimic:")
total = 0
for col, count in fill_stats.items():
    if count > 0:
        print(f"  ✓ {col}: +{count}")
        total += count

print(f"\nTotal values added: {total}")

# Verify key columns
print(f"\nVerification:")
print(f"  Unique patients: {df_merged['stay_id'].nunique()}")
print(f"  avg_hgb non-null: {df_merged['avg_hgb'].notna().sum()}")
print(f"  combined_vaso non-zero: {(df_merged['combined_vaso'] > 0).sum()}")

# Assign to df
df = df_merged

print("\n✓ Done! 'df' is now in 6h bins with fused data from mimic")

In [ ]:
#fluids_ml (daily sum, mean) too high x10
#blood_input (daily sum, mean) too low x10
#colloids_ml (daily sum, mean) too low x10
#harnk_ml (daily sum, mean) too high x10


In [ ]:
df

In [ ]:
df['avg_hgb'].value_counts()

In [ ]:
df_raw['avg_hgb'].value_counts()

In [ ]:
df['avg_hgb'].value_counts()

In [ ]:
df_raw['total_urineoutput'].value_counts()

In [ ]:
df['total_urineoutput'].value_counts()

In [ ]:
df

In [ ]:
# Filter to patients with at least one hb_combined measurement
df = df.groupby('stay_id').filter(lambda x: len(x) >= 4)

# Remove IDs where there is no single entry in column hemoglobin_g/dl
df = df[df.groupby('stay_id')['avg_hgb'].transform('count') > 1]

In [ ]:
df

In [ ]:
df['stay_id'].nunique()

In [ ]:
# 1. Rename columns
df = df.rename(columns={
    'stay_id': 'encounterId',
    'chartdate': 'utcChartTime' ,
    'round': 'age',
    'gender' : 'sex_or_gender'
})



# 3. Drop unwanted cols
df = df.drop(columns=[
    'subject_id', 'hadm_id',
    'icu_intime', 'icu_outtime', 'window_end'
])

# 4. Rename all the “avg_…” and drain/input columns to your numeric_columns names
numeric_mapping = {
    # vital‐signs
    'avg_sbp':      'blood_pressure_systolic_mmHg',
    'avg_mbp':      'blood_pressure_mean_mmHg',
    'avg_dbp':      'blood_pressure_diastolic_mmHg',
    'combined_vaso':     'combined_vaso',
    'crystalloid_input': 'fluids_ml',
    'colloid_input':     'colloids_ml',

    'avg_fibrinogen':   'fibrinogen_mg/dl',
    'avg_platelet':     'platelet_count_G/l',
    'max_lactate':      'lactate_mmol/l',
    'min_baseexcess':   'base_excess_mmol/l',
    'avg_hgb':          'hemoglobin_g/dl',
    'avg_spo2': 'spo2',
    'total_urineoutput':          'harnk_ml',
    'chest_pericardial_output':   'thorax-drain_ml',
    'jp_penrose_output':          'jackson-pratt_ml',
    'general_drain_output':       'redon-drain_ml',
    'hemo_wound_output':          'easy-flow_ml',
    'avg_resp_rate': 'respiratory_rate',
    'avg_heart_rate' : 'heart_rate',
    'blood_input' : 'blood_input'
}

df['robinson-drain_ml']= 0


df = df.rename(columns=numeric_mapping)


merged_df = df

In [ ]:
print('avg_mbp' in df.columns)


In [ ]:
merged_df.describe()

In [ ]:
# -------------------- Data Cleaning --------------------

# 1. Convert 'utcChartTime' to datetime
merged_df['utcChartTime'] = pd.to_datetime(merged_df['utcChartTime'], errors='coerce')

# 2. Convert numeric columns to appropriate types
numeric_columns = [
    'blood_pressure_systolic_mmHg', 'blood_pressure_mean_mmHg',
    'blood_pressure_diastolic_mmHg', 'combined_vaso', 'fluids_ml', 'colloids_ml',
    'fibrinogen_mg/dl', 'platelet_count_G/l',
    'lactate_mmol/l', 'base_excess_mmol/l', 'hemoglobin_g/dl',
    'harnk_ml', 'thorax-drain_ml', 'jackson-pratt_ml', 'redon-drain_ml',
    'easy-flow_ml', 'robinson-drain_ml', 'age', 'heart_rate',
    'respiratory_rate', 'spo2', 'blood_input', 
    'urine_output_ml'
]

for col in numeric_columns:
    if col in merged_df.columns:
        merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce')

# 4. Handle missing values
# Drop rows with missing 'encounterId' or 'utcChartTime'
merged_df = merged_df.dropna(subset=['encounterId', 'utcChartTime'])

# 5a. ARTIFACT RANGES: values outside these are set to NaN (assumed measurement error)

artifact_ranges = {
    'combined_vaso': (0.0, 10.0),       # >5 µg/kg/min very likely documentation / unit error
    'fluids_ml': (0.0, 200000.0),                 # extremely high to catch absurd values
    'colloids_ml': (0.0, 20000.0),
    'fibrinogen_mg/dl': (0.0, 4000.0),
    'platelet_count_G/l': (0.0, 2500.0),
    'lactate_mmol/l': (0.0, 50.0),
    'base_excess_mmol/l': (-50.0, 50.0),
    'urine_output_ml': (0.0, 25000.0),
    'thorax-drain_ml': (0.0, 20000.0),
    'jackson-pratt_ml': (0.0, 20000.0),
    'redon-drain_ml': (0.0, 20000.0),
    'easy-flow_ml': (0.0, 20000.0),
    'robinson-drain_ml': (0.0, 20000.0),
    'harnk_ml': (0.0, 20000.0),
    'age': (0.0, 130.0),                          # hard artifacts (negatives, ultra-high)
    'heart_rate': (0.0, 350.0),
    'respiratory_rate': (0.0, 200.0),
    'blood_input': (0.0, 20000.0),
    'spo2': (0.0, 110.0),

    'blood_pressure_systolic_mmHg': (20.0, 350.0),
    'blood_pressure_diastolic_mmHg': (10.0, 250.0),
    'blood_pressure_mean_mmHg': (20.0, 300.0),

    'hemoglobin_g/dl': (2.0, 30.0)
}

for col, (min_art, max_art) in artifact_ranges.items():
    if col in merged_df.columns:
        merged_df.loc[(merged_df[col] < min_art) | (merged_df[col] > max_art), col] = np.nan
        # Optional: keep an artifact flag instead of just NaN
        # merged_df[col + '_artifact'] = ((merged_df[col] < min_art) | (merged_df[col] > max_art)).astype(int)

# 5b. CLAMPING RANGES: for the remaining values, clamp into medically meaningful bounds


clamp_ranges = {
    'combined_vaso': (0.0,3.0),       # usual max doses
    'fluids_ml': (0.0, 3000.0),                  # still generous but tighter than artifact range
    'colloids_ml': (0.0, 400.0),
    'fibrinogen_mg/dl': (50.0, 1500.0),
    'platelet_count_G/l': (30.0, 1500.0),
    'lactate_mmol/l': (0.2, 20.0),
    'base_excess_mmol/l': (-20.0, 20.0),
    'urine_output_ml': (0.0, 400.0),
    'thorax-drain_ml': (0.0, 400.0),
    'jackson-pratt_ml': (0.0, 400.0),
    'redon-drain_ml': (0.0,400.0),
    'easy-flow_ml': (0.0, 400.0),
    'robinson-drain_ml': (0.0, 400.0),
    'harnk_ml': (0.0, 2000.0),
    'age': (18.0, 110.0),                         # if you’re only modelling adults
    'heart_rate': (20.0, 220.0),
    'respiratory_rate': (1.0, 70.0),
    'blood_input': (0.0, 2000.0),
    'spo2': (70.0, 100.0),

    'blood_pressure_systolic_mmHg': (60.0, 220.0),
    'blood_pressure_diastolic_mmHg': (30.0, 140.0),
    'blood_pressure_mean_mmHg': (40.0, 160.0),

    'hemoglobin_g/dl': (5.0, 20.0)
}

for col, (min_clamp, max_clamp) in clamp_ranges.items():
    if col in merged_df.columns:
        merged_df[col] = merged_df[col].clip(lower=min_clamp, upper=max_clamp)




In [ ]:
merged_df['harnk_ml'].describe()

In [ ]:
merged_df['harnk_ml'].dropna().describe()

In [ ]:
merged_df['drain_sum'] = merged_df['jackson-pratt_ml'] + merged_df['redon-drain_ml'] + merged_df['robinson-drain_ml'] + merged_df['thorax-drain_ml'] + merged_df['easy-flow_ml'] 
merged_df.drop(columns=['jackson-pratt_ml', 'redon-drain_ml', 'robinson-drain_ml', 'thorax-drain_ml','easy-flow_ml'])

In [ ]:
merged_df.columns

In [ ]:
cols_to_change_to_NaN = ['harnk_ml', 'drain_sum', 'blood_input', 'combined_vaso', 'colloids_ml', 'fluids_ml']
for col in cols_to_change_to_NaN:
    merged_df.loc[merged_df[col]==0,col] = np.NaN

In [ ]:

new_order = [
    'utcChartTime', 'encounterId', 'age', 'sex_or_gender',
    'blood_pressure_diastolic_mmHg', 'blood_pressure_mean_mmHg', 'blood_pressure_systolic_mmHg', 'spo2',
    'combined_vaso', 'colloids_ml', 'fluids_ml', 'fibrinogen_mg/dl',
    'lactate_mmol/l', 'platelet_count_G/l', 'hemoglobin_g/dl',
    'base_excess_mmol/l', 'harnk_ml', 'drain_sum', 'heart_rate', 'respiratory_rate', 'blood_input'
]


# reorder, raising if any are missing
merged_df = merged_df.reindex(columns=new_order)

In [ ]:
for col in merged_df.columns:
    print(col, merged_df[col].count())

In [ ]:
merged_df.loc[merged_df['hemoglobin_g/dl'].notna(),:]

In [ ]:
# Check the distribution before fixing
print("Min before fix:", merged_df['fibrinogen_mg/dl'].min())
print("Max before fix:", merged_df['fibrinogen_mg/dl'].max())

# Apply the conversion
# Logic: If value is less than 20, assume it is g/L and multiply by 100 to get mg/dL
mask_gl = (merged_df['fibrinogen_mg/dl'] < 100) & (merged_df['fibrinogen_mg/dl'] > 0)
merged_df.loc[mask_gl, 'fibrinogen_mg/dl'] = merged_df.loc[mask_gl, 'fibrinogen_mg/dl'] * 10

# Verify the fix
print("Min after fix:", merged_df['fibrinogen_mg/dl'].min())
print("Max after fix:", merged_df['fibrinogen_mg/dl'].max())

In [ ]:
cols_to_fix = ['harnk_ml', 'drain_sum']

for col in cols_to_fix:
    # Logic: If value is < 10, it is likely dL (or L). 
    # A value of 10 mL is very low, but 10 dL (1000ml) is high. 
    # So 10 is a safe separation point.
    
    mask_low = (merged_df[col] < 100)  & (merged_df[col] > 0) 
    
    # APPLY FACTOR 100 (Assuming dL -> mL)
    merged_df.loc[mask_low, col] = merged_df.loc[mask_low, col] * 6
    
    print(f"Corrected {col}. New description:")
    print(merged_df[col].describe())

In [ ]:
# =============================================================================
# BASE EXCESS PREPROCESSING - Clinical Approach
# =============================================================================

"""
Clinical reasoning for Base Excess (BE):

1. NORMAL RANGE: -2 to +2 mmol/L

2. INTERPRETATION:
   - Negative BE (< -2): Metabolic ACIDOSIS
     * Causes: Lactic acidosis, DKA, renal failure, diarrhea
     * Clinical significance: Indicates tissue hypoperfusion/shock
   
   - Positive BE (> +2): Metabolic ALKALOSIS
     * Causes: Vomiting, diuretics, hypokalemia, massive transfusion
     * Clinical significance: Usually less acute but still abnormal

3. WHY RANK TRANSFORM IS GOOD:
   - BE has a natural center around 0
   - Extreme values in BOTH directions are clinically important
   - Rank transform preserves the ordering without assuming distribution shape

4. ALTERNATIVE: Absolute deviation from normal
   - Create 'BE_deviation' = |BE - 0| 
   - This captures "how abnormal" regardless of direction
   - Useful if you care about severity, not direction
"""

def preprocess_base_excess(df):
    """
    Clinical preprocessing of base excess.
    """
    
    # Artifact removal: BE outside -30 to +30 is almost certainly error
    df.loc[(df['base_excess_mmol/l'] < -30) | 
           (df['base_excess_mmol/l'] > 30), 'base_excess_mmol/l'] = np.nan
    
    # Clinical clamp: -20 to +20 covers >99% of real values
    df['base_excess_mmol/l'] = df['base_excess_mmol/l'].clip(lower=-20, upper=20)
    

    return df


def impute_base_excess(df):
    """
    Imputation strategy for base excess.
    
    Clinical reasoning:
    - If BE is missing, the patient is likely stable (no urgent ABG needed)
    - Impute with 0 (normal) rather than median of sick population
    - Forward-fill makes sense for short gaps
    """
    
    # Forward fill first (recent value is most relevant)
    df['base_excess_mmol/l'] = df.groupby('encounterId')['base_excess_mmol/l'].ffill()
    
    # Fill remaining NaN with 0 (normal) rather than population median
    # Rationale: Missing ABG often means patient is stable
    df['base_excess_mmol/l'] = df['base_excess_mmol/l'].fillna(0)
    
    return df

In [ ]:

# Cell ~47: After cleaning, BEFORE imputation
# Apply base excess preprocessing:
df = preprocess_base_excess(df)


In [ ]:
# Select the same columns
cols_to_save = new_order + ['sex_or_gender'] 
# Assuming your external dataframe is named 'external_df'



merged_df[cols_to_save].to_csv('data_external.csv', index=False) 
print("External data saved.")

In [ ]:
for col in merged_df.columns:
    print(col, merged_df[col].count())

In [ ]:
# 1) Sort by encounter and time
merged_df = merged_df.sort_values(['encounterId', 'utcChartTime'])

# 2) Group columns by imputation strategy
med_rate_cols = [
    'combined_vaso'
]
vital_lab_cols = [
    'blood_pressure_systolic_mmHg', 'blood_pressure_mean_mmHg',
    'blood_pressure_diastolic_mmHg', 'spo2', 'fibrinogen_mg/dl', 
    'platelet_count_G/l', 'lactate_mmol/l', 'heart_rate', 'respiratory_rate',
    'base_excess_mmol/l'
]
sum_cols = [
    'fluids_ml', 'colloids_ml', 'harnk_ml', 'drain_sum', 'blood_input'
]

# 3) Fill volumes/sums with 0 (Missing volume = No volume given/output)
merged_df[sum_cols] = merged_df[sum_cols].fillna(0)

# 4) Forward-fill (ffill) within each encounter
# This maintains the last known state (e.g., last BP or last Med rate)
all_continuous = med_rate_cols + vital_lab_cols
merged_df[all_continuous] = (
    merged_df
    .groupby('encounterId', sort=False)[all_continuous]
    .ffill()
)

# 5) Handle remaining initial NaNs (the gaps before the first measurement)

# A: For Meds, initial NaN = 0 (The patient wasn't on the drug yet)
merged_df[med_rate_cols] = merged_df[med_rate_cols].fillna(0)

# B: For Vitals/Labs, initial NaN = Global Median (The patient is alive/average)
vitals_medians = merged_df[vital_lab_cols].median(numeric_only=True)
merged_df[vital_lab_cols] = merged_df[vital_lab_cols].fillna(vitals_medians)

In [ ]:
merged_df

In [ ]:
merged_df.to_parquet(os.path.join(data_path, f'mimic_df.parquet'))

In [ ]:
merged_df.columns

In [ ]:
merged_df

In [ ]:
# Identify encounterIds that have at least one non-null value in 'hemoglobin_g/dl'
valid_encounters = merged_df.dropna(subset=['hemoglobin_g/dl'])['encounterId'].unique()

# Filter the original dataframe to keep only those encounterIds
filtered_df = merged_df[merged_df['encounterId'].isin(valid_encounters)]

In [ ]:
df = filtered_df

In [ ]:
df = filtered_df

In [ ]:
features = med_rate_cols+vital_lab_cols+vital_lab_cols+sum_cols
for col in features:
    
    print(col)
    print(df[col].describe())
    print(df.loc[df[col]>0,col].describe())
    print('___________________________________ \n')
